In [ ]:
# Scam type
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

scam_stats_df = pd.read_csv('scam_category_stats.csv')
scam_stats_sorted = scam_stats_df.sort_values('Counts', ascending=False).reset_index(drop=True)
top_n = 30
top_scams = scam_stats_sorted.head(top_n).copy()
visual_scam_data = top_scams
colors = plt.cm.gist_rainbow(np.linspace(0, 1, len(visual_scam_data)))
# ======环形饼图======
fig1, ax1 = plt.subplots(figsize=(12, 10))
pie_data = visual_scam_data.copy()
wedges, texts, autotexts = ax1.pie(
    pie_data['Proportion(%)'], 
    autopct='%1.1f%%',      
    colors=colors,
    startangle=90,
    pctdistance=0.85,       
    wedgeprops=dict(width=0.3, edgecolor='white')
)
ax1.set_title(f'Scam Type Proportion Distribution (Top {top_n})', fontsize=14, fontweight='bold', pad=20)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(8)
plt.legend(
    wedges, 
    pie_data['Scam_type'],
    title="Scam Categories",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1), 
    fontsize=9
)
plt.tight_layout()
plt.savefig('Scam Type Proportion Distribution_Top30.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# Topic clustering
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('讨论主题cluster_topics.csv')
df = df[df['clustered_topic'] != 'other']
 
# ======全平台 Top20 热门主题水平条形图======
topic_counts = df['clustered_topic'].value_counts()
top20 = topic_counts.head(20)
plt.figure(figsize=(12, 8))
colors = sns.color_palette("viridis", n_colors=20)
plt.barh(top20.index[::-1], top20.values[::-1], color=colors)
plt.xlabel('Number of Comments')
plt.title('Top 20 Hot Topics Across All Platforms')
plt.tight_layout()
plt.savefig('top20_topics.png', dpi=150)
plt.close()

# ======五个平台热门话题======
platforms = ['Reddit', 'PTT', 'FACEBOOK', 'LIHKG', 'Threads']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten() 
for i, platform in enumerate(platforms):
    pf_data = df[df['platform'] == platform]
    if pf_data.empty:
        axes[i].set_visible(False)
        continue
    pf_topics = pf_data['clustered_topic'].value_counts().head(10)
    colors = sns.color_palette("Set2", n_colors=10)
    axes[i].barh(pf_topics.index[::-1], pf_topics.values[::-1], color=colors)
    axes[i].set_title(f'Top Topics in {platform}', fontsize=12)
    axes[i].set_xlabel('Count')
    axes[i].tick_params(axis='y', labelsize=9)
axes[5].set_visible(False)
plt.suptitle('Platform-wise Hot Topics Comparison'x, fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('platform_topics.png', dpi=150)
plt.close()

In [ ]:
# victim group
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
df = pd.read_csv("受害群体划分cluster_victim.csv")
dimensions = {
    'Gender': 'victim_gender_clustered',
    'Age Group': 'victim_age_group_clustered',
    'Occupation': 'victim_occupation_clustered',
    'Other Identity': 'victim_other_identity_clustered'
}
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()
titles = {
    'Gender': 'Gender Distribution',
    'Age Group': 'Age Group Distribution',
    'Occupation': 'Top 20 Occupations',
    'Other Identity': 'Top 20 Special Identity'
}
for idx, (dim_name, col) in enumerate(dimensions.items()):
    counts = df[col].value_counts()
    counts = counts[~counts.index.isin(['Unspecified', 'Unknown'])]
    if dim_name in ['Occupation', 'Other Identity']:
        counts = counts.head(20)
    if len(counts) == 0:
        axes[idx].set_title(f'{dim_name} (No Data)')
        continue

    counts = counts.sort_values(ascending=True)
    colors = plt.cm.tab20.colors[:len(counts)]
    bars = axes[idx].barh(counts.index, counts.values, color=colors)
    axes[idx].set_title(titles[dim_name], fontsize=14)
    axes[idx].set_xlabel('Number of Victims', fontsize=11)
    for bar, val in zip(bars, counts.values):
        axes[idx].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, str(val),
                       va='center', fontsize=9)
plt.tight_layout(pad=3.0)
plt.savefig('victim_profile_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
# Gender x Scam type

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
from json.decoder import JSONDecodeError

plt.rcParams['axes.unicode_minus'] = False
plt.style.use('default')
df = pd.read_csv('gender x scam type.csv')

def parse_json_scam(scam_str):
    if pd.isna(scam_str) or scam_str == '':
        return []  
    try:
        return json.loads(scam_str.replace("'", '"')) 
    except JSONDecodeError:
        return [] 

df['scam_list'] = df['main_scam_category'].apply(parse_json_scam)
df_exploded = df.explode('scam_list', ignore_index=True)
df_exploded = df_exploded[df_exploded['scam_list'] != ''].copy()
df_exploded['gender_clean'] = df_exploded['gender'].astype(str).str.strip().str.lower()
df_exploded['scam_clean'] = df_exploded['scam_list'].astype(str).str.strip().str.lower()

df_filtered = df_exploded[
    (df_exploded['gender_clean'].isin(['female', 'male'])) 
    & (~df_exploded['scam_clean'].str.contains('other', na=False))  
].copy()

top_scam_counts = df_filtered['scam_list'].value_counts().head(10)
top_scam_types = top_scam_counts.index.tolist()
df_top_scam = df_filtered[df_filtered['scam_list'].isin(top_scam_types)].copy()

cross_tab = pd.crosstab(
    index=df_top_scam['gender'], 
    columns=df_top_scam['scam_list'], 
    margins=False  
)
fig, ax = plt.subplots(figsize=(16, 8))
heatmap = sns.heatmap(
    cross_tab,
    annot=True, 
    fmt='d', 
    cmap='YlOrRd',  
    cbar_kws={'label': 'Case counts'}, 
    ax=ax,
    linewidths=0.8,  
    vmin=0, 
    annot_kws={'fontsize': 9} 
)

ax.set_xlabel('Scam types', fontsize=13, fontweight='bold', labelpad=12)
ax.set_ylabel('Victim Gender', fontsize=13, fontweight='bold', labelpad=12)
ax.set_title('Cross Heatmap of Victim Gender × Scam Type', 
             fontsize=15, fontweight='bold', pad=25)

plt.xticks(rotation=45, ha='right', fontsize=10, rotation_mode='anchor')
plt.yticks(fontsize=11)
plt.tight_layout()
plt.savefig('gender_scam_json_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
